# 🎯 Análise de Centralidade - Game of Thrones
## Quem é matematicamente o personagem mais importante?

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import numpy as np

In [ ]:
df = pd.read_csv("../datasets/interacoes.csv")
df_direct = df[df["tipo_interacao"] == "single"]
df_grouped = df_direct.groupby(['falante', 'ouvinte']).size().reset_index(name='weight')
G = nx.from_pandas_edgelist(df_grouped, 'falante', 'ouvinte', edge_attr='weight', create_using=nx.Graph())
print(f"✅ Grafo: {G.number_of_nodes()} nós, {G.number_of_edges()} arestas")

## 1️⃣ Centralidade de Grau

In [ ]:
degree_cent = nx.degree_centrality(G)
top_degree = sorted(degree_cent.items(), key=lambda x: x[1], reverse=True)[:10]
pd.DataFrame(top_degree, columns=['Personagem', 'Score'])

## 2️⃣ Centralidade de Intermediação

In [ ]:
betweenness_cent = nx.betweenness_centrality(G, weight='weight')
top_betweenness = sorted(betweenness_cent.items(), key=lambda x: x[1], reverse=True)[:10]
pd.DataFrame(top_betweenness, columns=['Personagem', 'Score'])

## 3️⃣ PageRank

In [ ]:
pagerank = nx.pagerank(G, weight='weight')
top_pagerank = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)[:10]
pd.DataFrame(top_pagerank, columns=['Personagem', 'Score'])

## 4️⃣ Eigenvector Centrality

In [ ]:
try:
    eigenvector_cent = nx.eigenvector_centrality(G, weight='weight', max_iter=1000)
except:
    eigenvector_cent = nx.eigenvector_centrality(G, max_iter=1000)
top_eigenvector = sorted(eigenvector_cent.items(), key=lambda x: x[1], reverse=True)[:10]
pd.DataFrame(top_eigenvector, columns=['Personagem', 'Score'])

## 5️⃣ Weighted Degree

In [ ]:
weighted_degree = dict(G.degree(weight='weight'))
top_weighted = sorted(weighted_degree.items(), key=lambda x: x[1], reverse=True)[:10]
pd.DataFrame(top_weighted, columns=['Personagem', 'Interações'])

## 🏆 Ranking Consolidado
**Pesos:** PageRank(40%), Betweenness(30%), Weighted Degree(20%), Eigenvector(10%)

In [ ]:
personagens = list(degree_cent.keys())
scaler = MinMaxScaler()

norm_pr = scaler.fit_transform(np.array([pagerank[p] for p in personagens]).reshape(-1,1)).flatten()
norm_bt = scaler.fit_transform(np.array([betweenness_cent[p] for p in personagens]).reshape(-1,1)).flatten()
norm_wd = scaler.fit_transform(np.array([weighted_degree[p] for p in personagens]).reshape(-1,1)).flatten()
norm_ev = scaler.fit_transform(np.array([eigenvector_cent.get(p, 0) for p in personagens]).reshape(-1,1)).flatten()

scores = {p: norm_pr[i]*0.4 + norm_bt[i]*0.3 + norm_wd[i]*0.2 + norm_ev[i]*0.1 for i,p in enumerate(personagens)}
top = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:20]

df_final = pd.DataFrame([
    (p, scores[p], pagerank[p], betweenness_cent[p], weighted_degree[p], eigenvector_cent.get(p, 0)) 
    for p,_ in top
], columns=["Personagem","Score","PageRank","Betweenness","W_Degree","Eigenvector"])

df_final

## 📊 Visualização

In [ ]:
plt.figure(figsize=(14,8))
plt.barh(df_final["Personagem"], df_final["Score"], color="darkred")
plt.xlabel("Score Consolidado")
plt.title("Top 20 Personagens Mais Importantes - Game of Thrones")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()